제한 및 규칙

- 파인튜닝(Fine-tuning) 금지: 베이스 모델은 동결(requires_grad = False) 상태 유지
- 베이스 모델 자유 선택: torchvision.models 계열(ResNet, EfficientNet, MobileNetV2, VGG 등) 중 택 1 이상
- 데이터 로드/증강은 힌트만 제공(학생이 직접 구현)
- 캐글(Kaggle) 등에서 2개 ~ 6개 내외의 클래스를 가진 이미지 분류 데이터셋을 자유롭게 선정하시오.

 제출물(필수)

1. 실행 파일(.py 또는 .ipynb)
2. 학습 로그(에폭별 train/val loss, accuracy)
3. best 모델 파일(Checkpoint로 저장된 모델)
4. 평가 리포트(MD/PDF):

- 베이스 모델 선택 이유
- 최종 성능 지표(Accuracy + 추가 지표 1개)
- 혼동행렬 + 오분류 분석(5개 이상)
- 개선 아이디어 2개(파인튜닝 제외)

In [3]:
import os
import time
import platform
from shutil import rmtree
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as image
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

# 1. 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':  # Mac OS
    plt.rc('font', family='AppleGothic')
else:  # Linux (Colab 등)
    plt.rc('font', family='NanumBarunGothic')

# 2. 마이너스 기호(-) 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

load_dotenv()

# 오류 발생시
# sklearn.set_config(display="text")

True

[문제 1] 데이터 로드 & 증강 파이프라인 구성 (힌트 제공) (15점)

요구사항

- 입력 크기: 224 x 224
- 배치 크기: 32
- 훈련/검증 분리: 80 / 20
- 훈련(train)에만 증강 적용
- 클래스 수: 본인이 선택한 데이터셋 기준으로 명시 (예: 2진 분류 또는 다중 분류)
  
힌트(제공)

- torchvision.datasets.ImageFolder 및 random_split 활용 가능
- torchvision.transforms를 사용하여 훈련셋에는 Resize, Random 변환, ToTensor, Normalize 적용
- 검증셋에는 Resize, ToTensor, Normalize 적용

채점 기준(15점)

- (6점) 80/20 분리 + DataLoader 파이프라인 정상 동작
- (5점) train만 증강, val은 증강 없이 전처리만
- (4점) 클래스 수/라벨 매핑/샘플 수 확인 결과 제시

## 커피 원두 상태 분류

### 커피 원두 데이터셋
- 커피 원두의 색상과 표면 특징을 이용해 종류나 로스팅 상태를 구분합니다. 
- 선정 과제: 원두 이미지로 로스팅 단계 예측하기
- https://www.kaggle.com/datasets/gpiosenka/coffee-bean-dataset-resized-224-x-224

In [26]:
base_path = r"/Users/hyusoohi5/kdt_0900_khs/ml/resource/data_set"

In [27]:
coffee_bean_path = os.path.join(base_path, "coffee_beans")
os.makedirs(coffee_bean_path, exist_ok=True)

api = KaggleApi()
api.authenticate()

os.makedirs(base_path, exist_ok=True)
api.dataset_download_files(r"gpiosenka/coffee-bean-dataset-resized-224-x-224", path=base_path, unzip=True)

print("다운로드 완료")

Dataset URL: https://www.kaggle.com/datasets/gpiosenka/coffee-bean-dataset-resized-224-x-224
다운로드 완료


In [31]:
train_path = os.path.join(base_path, r"coffee_beans", r"train")
test_path = os.path.join(base_path, r"coffee_beans", r"test")

In [32]:
base_dataset = ImageFolder(root=coffee_bean_path, transform=None)
base_dataset

Dataset ImageFolder
    Number of datapoints: 1600
    Root location: /Users/hyusoohi5/kdt_0900_khs/ml/resource/data_set/coffee_beans

### Data Split

In [17]:
total_size = len(base_dataset)
train_size = int(total_size * 0.5)
test_size = total_size - train_size

In [19]:
print(train_size, test_size)

800 800


In [20]:
train_subset, test_subset = random_split(base_dataset, [train_size, test_size])

### Transform

In [33]:
# 1. transform 설정

train_transform = transforms.Compose([
    # 모든 이미지 값들의 사이즈가 ((224, 224))여서 따로 Resize()하지 않았습니다.
    # ToTensor()
    transforms.ToTensor(),
    # Normalize()
    transforms.Normalize(
        mean = (0.485, 0.456, 0.406),
        std = (0.229, 0.224, 0.225)
    ),
])

test_transform = transforms.Compose([
    # 모든 이미지 값들의 사이즈가 ((224, 224))여서 따로 Resize()하지 않았습니다.
    # ToTensor()
    transforms.ToTensor(),
    # Normalize()
    transforms.Normalize(
        mean = (0.485, 0.456, 0.406),
        std = (0.229, 0.224, 0.225)
    ),
])

In [34]:
class DatasetWrapper(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

In [35]:
train_dataset = ImageFolder(train_path, transform=train_transform)
test_dataset = ImageFolder(test_path, transform=test_transform)

## Data Loader

In [36]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [38]:
imgs, labels = next(iter(train_dataloader))

print("이미지 shape :", imgs.shape)
print("라벨 shape :", labels.shape)
print("라벨 dtype :", labels.dtype)

이미지 shape : torch.Size([32, 3, 224, 224])
라벨 shape : torch.Size([32])
라벨 dtype : torch.int64


In [39]:
model1 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

In [40]:
model1

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

[문제 2] 베이스 모델 선택 & 선택 이유(서술) (15점)

요구사항 다음 조건을 만족하도록 베이스 모델 1개를 선택하고, 선택 이유를 8~12줄로 서술하라. 서술 항목(필수 포함)

1. 데이터 특성(클래스 수, 데이터 규모 추정, 난이도, 이미지 특징)
2. 모델 트레이드오프(정확도 vs 속도 vs 모델 크기/파라미터 수)
3. 실행 환경 고려(로컬 GPU/MPS, Colab T4, 학습 시간 등)
4. 위험요소(과적합, 배경 편향, 전처리 호환 등)와 대응 1개

채점 기준(15점)

- (6점) 데이터 특성과 모델 선택이 논리적으로 연결
- (5점) 성능/속도/용량 트레이드오프 구체 언급
- (4점) 환경/시간/위험요소까지 고려

[문제 3] 전이학습 모델 구현 & 훈련(Feature Extraction) (40점)

요구사항(공통)

선택한 베이스 모델로 특징추출 전이학습 모델을 구성하고 학습하라.

필수 조건

- 사전 모델 상위 분류기 제거 및 교체
- base model은 동결(requires_grad = False) 유지
- 분류기 헤드(필수):
- nn.AdaptiveAvgPool2d((1, 1)) (GAP 적용)
- nn.Dropout(권장)
- nn.Linear(in_features, num_classes)
- optimizer: Adam, learning_rate=0.001
- 체크포인트:
- 검증 정확도 기준 최고 성능 갱신 시 torch.save로 best 모델 저장

채점 기준(40점)

- (10점) base model 로딩 + 입력 형태 일치
- (10점) base_model 동결(requires_grad=False) 정확히 적용
- (8점) GAP 기반 헤드 구성 적절(Flatten 위주 구성은 감점)
- (6점) 손실함수/옵티마이저/학습률 설정 적절
- (6점) 최고 성능 갱신형 가중치 저장(torch.save) 로직 무결성

[문제 4] 모델 평가 및 측정 리포트 (30점)

요구사항

저장된 best 모델을 불러와 아래 항목을 수행하고 리포트로 작성하라.

필수 항목

1. Accuracy 측정(검증 또는 테스트)
2. 추가 지표 1개 선택(Precision/Recall/F1 중 택1 권장)
3. 혼동행렬(confusion matrix) 생성
4. 오분류 5개 이상 추출

- 각 샘플: 정답 라벨 / 예측 라벨 / 예측 확률(또는 점수) 포함

5. 개선 아이디어 2개 제시(파인튜닝 제외)
- 예: 증강 강화, Dropout 조정, 입력 크기/배치 조정, 데이터 불균형 대응 등
- 왜 도움이 되는지 근거 1줄 포함

채점 기준(30점)

- (8점) best 모델 로드 + 정확도 측정 재현 가능
- (6점) 추가 지표 계산 및 해석
- (8점) 혼동행렬 해석이 구체적(어떤 클래스끼리 혼동하는지)
- (8점) 오분류 5개 이상 + 확률/점수 근거 포함

[보너스] Grad-CAM 시각화로 “모델이 어디를 봤는지” 설명 (+10점)

요구사항 best 모델의 마지막 컨볼루션 층(또는 적절한 레이어)에 훅을 부착하여 Grad-CAM을 생성하고 모델의 판단 근거를 설명하라.

제출 조건 - 정분류 3장 + 오분류 2장 이상(총 5장 이상)

- 각 이미지마다 제출:
- 원본 이미지
- Grad-CAM 히트맵(overlay)
- 한 줄 해석: “모델이 어디를 보고 왜 그렇게 예측했는지”
- 오분류 2장에 대해 추가:
- 오분류 원인 추정(배경 편향/유사 패턴/노이즈 등)
- 개선 아이디어 1개(파인튜닝 제외) 채점 기준(+10점)

- (3점) 최소 5장 충족 + 시각화 품질(overlay 포함)
- (3점) 관찰이 구체적(대상 부위/특징을 정확히 언급)
- (2점) 오분류 원인 분석 타당 - (2점) 개선 아이디어가 원인과 연결